# 🎙️ ASR Training — Bahasa Indonesia (CNN 1D)
## Proyek: SuaraKita — Automatic Speech Recognition

**Pipeline:**
1. Mount Google Drive & Upload Dataset
2. Preprocessing Audio
3. Ekstraksi Fitur MFCC
4. Training Model CNN 1D
5. Evaluasi Model
6. Simpan model ke `model_asr.h5`


In [ ]:
# ── Cell 1: Install & Import ─────────────────────────────────────────
!pip install librosa soundfile -q

import os
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0D1117'
matplotlib.rcParams['axes.facecolor']   = '#161B22'
matplotlib.rcParams['text.color']       = '#E6EDF3'
matplotlib.rcParams['axes.labelcolor']  = '#8B949E'
matplotlib.rcParams['xtick.color']      = '#8B949E'
matplotlib.rcParams['ytick.color']      = '#8B949E'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')


In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path dataset di Google Drive kamu
DATASET_DIR = '/content/drive/MyDrive/speech-app/asr/dataset'
MODEL_SAVE_PATH = '/content/drive/MyDrive/speech-app/asr/model/model_asr.h5'

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
print(f'Dataset dir: {DATASET_DIR}')
print(f'Model save: {MODEL_SAVE_PATH}')


In [ ]:
# ── Cell 3: Konfigurasi ──────────────────────────────────────────────
LABELS = [
    'yesus', 'simon', 'andreas', 'yakobus', 'yohanes',
    'filipus', 'bartomeleus', 'tomas', 'matius',
    'tadeus', 'yudas', 'maria'
]
NUM_CLASSES = len(LABELS)
SAMPLE_RATE = 16000
DURATION    = 2.0       # detik
N_MFCC      = 13
MAX_LEN     = 32        # jumlah frame MFCC
BATCH_SIZE  = 32
EPOCHS      = 50

print(f'Jumlah kelas: {NUM_CLASSES}')
print(f'Label: {LABELS}')


In [ ]:
# ── Cell 4: Preprocessing & Ekstraksi MFCC ──────────────────────────
def load_and_preprocess(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """Load, normalize, trim, dan pad/truncate audio."""
    audio, _ = librosa.load(file_path, sr=sr, mono=True)
    # Normalize
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val
    # Trim silence
    audio, _ = librosa.effects.trim(audio, top_db=20)
    # Pad atau truncate
    target_len = int(sr * duration)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]
    return audio


def extract_mfcc(audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, max_len=MAX_LEN):
    """Ekstrak MFCC dan normalisasi shape."""
    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=n_mfcc,
        n_fft=512, hop_length=512, n_mels=40
    )
    mfcc = mfcc.T  # (T, n_mfcc)
    if mfcc.shape[0] < max_len:
        mfcc = np.pad(mfcc, ((0, max_len - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:max_len, :]
    return mfcc


def load_dataset(dataset_dir):
    X, y = [], []
    for label_idx, label_name in enumerate(LABELS):
        label_path = os.path.join(dataset_dir, label_name)
        if not os.path.exists(label_path):
            print(f'[WARN] Folder tidak ditemukan: {label_path}')
            continue
        wav_files = [f for f in os.listdir(label_path)
                     if f.lower().endswith('.wav')]
        print(f'  [{label_name}] → {len(wav_files)} file')
        for wav_file in wav_files:
            file_path = os.path.join(label_path, wav_file)
            try:
                audio = load_and_preprocess(file_path)
                mfcc  = extract_mfcc(audio)
                X.append(mfcc)
                y.append(label_idx)
            except Exception as e:
                print(f'  [ERROR] {wav_file}: {e}')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print('Loading dataset...')
X, y = load_dataset(DATASET_DIR)
print(f'\n✅ Dataset dimuat!')
print(f'X shape: {X.shape}   y shape: {y.shape}')
print(f'Distribusi kelas:')
for i, label in enumerate(LABELS):
    count = np.sum(y == i)
    bar = '█' * (count // 2)
    print(f'  {label:15s} {count:3d} {bar}')


In [ ]:
# ── Cell 5: Visualisasi MFCC Sampel ─────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(16, 10), facecolor='#0D1117')
fig.suptitle('MFCC Samples per Kelas', color='#F0F6FC',
             fontsize=14, fontweight='bold', y=1.01)

for i, label in enumerate(LABELS):
    idx = np.where(y == i)[0]
    if len(idx) == 0:
        continue
    sample_mfcc = X[idx[0]].T  # (N_MFCC, MAX_LEN)
    ax = axes[i // 4][i % 4]
    ax.set_facecolor('#161B22')
    im = ax.imshow(sample_mfcc, aspect='auto', origin='lower',
                   cmap='magma', interpolation='nearest')
    ax.set_title(label.capitalize(), color='#E6EDF3', fontsize=10, fontweight='bold')
    ax.set_xlabel('Frame', color='#8B949E', fontsize=8)
    ax.set_ylabel('MFCC', color='#8B949E', fontsize=8)
    ax.tick_params(colors='#8B949E', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363D')

plt.tight_layout()
plt.savefig('/content/mfcc_samples.png', dpi=120, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print('✅ Visualisasi MFCC disimpan')


In [ ]:
# ── Cell 6: Split Dataset ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f'Train: {X_train.shape[0]} sampel')
print(f'Val:   {X_val.shape[0]}   sampel')
print(f'Test:  {X_test.shape[0]}  sampel')
print(f'Input shape: {X_train.shape[1:]}')


In [ ]:
# ── Cell 7: Bangun Model CNN 1D ──────────────────────────────────────
def build_model(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv1D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),

        # Block 2
        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),

        # Block 3
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),

        # Dense
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ], name='ASR_CNN1D')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


input_shape = (MAX_LEN, N_MFCC)  # (32, 13)
model = build_model(input_shape, NUM_CLASSES)
model.summary()


In [ ]:
# ── Cell 8: Training ─────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        '/content/best_model.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training selesai!')


In [ ]:
# ── Cell 9: Plot Training History ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0D1117')

for ax in axes:
    ax.set_facecolor('#161B22')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363D')
    ax.grid(True, color='#21262D', linewidth=0.5)

# Accuracy
axes[0].plot(history.history['accuracy'],     color='#3B82F6', linewidth=2, label='Train')
axes[0].plot(history.history['val_accuracy'], color='#10B981', linewidth=2,
             linestyle='--', label='Validation')
axes[0].set_title('Model Accuracy', color='#E6EDF3', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch', color='#8B949E')
axes[0].set_ylabel('Accuracy', color='#8B949E')
axes[0].legend(facecolor='#1C2537', edgecolor='#30363D', labelcolor='#E6EDF3')

# Loss
axes[1].plot(history.history['loss'],     color='#F43F5E', linewidth=2, label='Train')
axes[1].plot(history.history['val_loss'], color='#F59E0B', linewidth=2,
             linestyle='--', label='Validation')
axes[1].set_title('Model Loss', color='#E6EDF3', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch', color='#8B949E')
axes[1].set_ylabel('Loss', color='#8B949E')
axes[1].legend(facecolor='#1C2537', edgecolor='#30363D', labelcolor='#E6EDF3')

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=120, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()


In [ ]:
# ── Cell 10: Evaluasi & Confusion Matrix ────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss:     {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print('\nClassification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=[l.capitalize() for l in LABELS]
))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(12, 10), facecolor='#0D1117')
ax.set_facecolor('#161B22')
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[l.capitalize() for l in LABELS],
    yticklabels=[l.capitalize() for l in LABELS],
    ax=ax,
    linewidths=0.5, linecolor='#0D1117'
)
ax.set_title('Confusion Matrix', color='#E6EDF3', fontsize=14, fontweight='bold', pad=16)
ax.set_xlabel('Predicted', color='#8B949E', fontsize=11)
ax.set_ylabel('True Label', color='#8B949E', fontsize=11)
ax.tick_params(colors='#8B949E')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()


In [ ]:
# ── Cell 11: Simpan Model ────────────────────────────────────────────
model.save(MODEL_SAVE_PATH)
print(f'✅ Model disimpan ke: {MODEL_SAVE_PATH}')

# Verifikasi
loaded = keras.models.load_model(MODEL_SAVE_PATH)
v_loss, v_acc = loaded.evaluate(X_test, y_test, verbose=0)
print(f'✅ Verifikasi model loaded — Accuracy: {v_acc*100:.2f}%')
print(f'\nSalin file model_asr.h5 ke folder: speech-app/asr/model/')
